In [1]:
print("Hello world")

Hello world


In [2]:
import sys

# Check if we are currently running inside Google Colab
if 'google.colab' in sys.modules:
    print("Colab environment detected. Installing dependencies and cloning repository...")
    # Major project dependencies used by this notebook and local modules.
    colab_requirements = [
        'numpy',
        'sentencepiece',
        'datasets',
        'jax[cpu]',
        'matplotlib',
        'ipykernel',
    ]

    !pip install -q {" ".join(colab_requirements)}
    
    # Run the shell command to clone
    !git clone https://github.com/krishnan1159/NMT.git
    
    # Add the cloned repo to the system path
    sys.path.append('/content/NMT')
    
else:
    print("Local environment detected. Skipping git clone.")

Colab environment detected. Installing dependencies and cloning repository...
fatal: destination path 'NMT' already exists and is not an empty directory.


In [3]:
if 'google.colab' not in sys.modules:
    %load_ext autoreload
    %autoreload 2
    %reload_ext autoreload

## Local classes
import importlib
from vocab import Tokenizer
from model_embedding import ModelEmbedding
from config import get_config
import rnn

importlib.reload(rnn)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


<module 'rnn' from '/content/NMT/rnn.py'>

In [4]:
from datasets import load_dataset
import jax
import jax.numpy as jp

In [5]:
ds = load_dataset("shiyue/chr_en", "parallel")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [6]:
train_set = list(ds["train"]["sentence_pair"])
chr_sentences = [sentence['chr'] for sentence in train_set]
eng_sentences = [sentence['en'] for sentence in train_set]

## Generate vocab for both source and target sentence.
cfg = get_config()
chr_vocab = Tokenizer(chr_sentences, "chr", "chr", "20000")
eng_vocab = Tokenizer(eng_sentences, "eng", "eng", "8000")
random_key = jax.random.key(0)
embedding = ModelEmbedding(chr_vocab.get_vocab_size(), eng_vocab.get_vocab_size(), cfg.embed_size, random_key)

In [7]:
print(eng_sentences[0])
a, b = eng_vocab.to_numpy([eng_sentences[1]])
print(b)
print(a.shape)
embedding.target_embed(a).shape

and by him every one that believeth is justified from all things, from which ye could not be justified by the law of Moses.
[29]
(1, 29)


(1, 29, 128)

In [11]:
chr_sentences = chr_sentences[:140]
eng_sentences = eng_sentences[:140]

chr_tokens, chr_lengths = chr_vocab.to_numpy(chr_sentences, add_bos=True, add_eos=True)
eng_tokens, eng_lengths = eng_vocab.to_numpy(eng_sentences, add_bos=True, add_eos=True)
print(eng_vocab.get_vocab_size())
print(cfg.num_epochs)

rnn.train(chr_tokens, chr_lengths, eng_tokens, eng_lengths, embedding, eng_vocab.get_vocab_size(), cfg)

INFO:rnn:JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
INFO:rnn:JAX backend: tpu
INFO:rnn:RNN cumulative params : embedding = 3584000, encoder = 32769, decoder = 1056769 and total = 4673538
INFO:rnn:Loss at epoch 1/10 is 8.987151145935059. Time taken to train = 0.005383759999972426
INFO:rnn:Loss at epoch 2/10 is 8.986983299255371. Time taken to train = 0.005454119999967588
INFO:rnn:Loss at epoch 3/10 is 8.986673355102539. Time taken to train = 0.005333889999974417
INFO:rnn:Loss at epoch 4/10 is 8.985990524291992. Time taken to train = 0.005202709999934996
INFO:rnn:Loss at epoch 5/10 is 8.984444618225098. Time taken to train = 0.005142960000057428
INFO:rnn:Loss at epoch 6/10 is 8.980935096740723. Time taken to train = 0.005267430000003515
INFO:rnn:Loss at epoch 7/10 is 8.972986221313477. Time taken to train = 0.00522365099993749
INFO:rnn:Loss at epoch 8/10 is 8.95494556427002. Time taken to train = 0.005199999999945248
INFO:rnn:Loss at epoch 9/10 is 8.9

8000
10
